In [1]:
import torch
import qewton
import plotly.io as pio

from load_data import (
    domain_var,
    temperature_constraint,
    temperature_var,
    x_var,
    load_data,
)

pio.renderers.default = "vscode"

In [2]:
batch_size = qewton.optim.DiscreteHyperparameter((5, 50))

dataset, data_loader = load_data(batch_size)


--- Data quality report ---
Total samples:     581
Valid samples:     580
Discarded samples: 1

Discarded samples:
  Sample 40: pressure: max abs = 8.7486e+14 (limit 1000000.0)


In [3]:
model = qewton.algorithms.CNNDeepONet(
    branch_input=domain_var,
    trunk_input=x_var,
    output=temperature_var,

    # CNN branch
    channels=(16, 32),
    conv_kernel_size=(3, 3, 3),
    pooling_kernel_size=2,
    pooling_type="average",

    # Trunk network
    trunk_hidden_neurons=64,
    trunk_hidden_layers=3,

    # Shared representation dimension
    intermediate_neurons=64,

    # FCN after convolutional encoder
    branch_fcn_hidden_neurons=64,
    branch_fcn_hidden_layers=1,
)

In [4]:
import qewton.algorithms.building_blocks.array_operations as array_ops
import inspect

print([
    name for name, obj in inspect.getmembers(array_ops, inspect.isclass)
])

['Annotated', 'AxesDim', 'ConcatNode', 'ConcatVariables', 'DEFAULT_DL_BACKEND', 'DataConfiguration', 'DeepLearningBackend', 'EllipsisAxes', 'EllipsisType', 'FeatureAxes', 'Flatten', 'GetShapeNode', 'InputPort', 'NO_DEFAULT', 'Narrow', 'Node', 'OutputPort', 'Port', 'Repeat', 'Reshape', 'ReshapeAtDim', 'SetItem', 'Slice', 'SplitVariables', 'Squeeze', 'StructureUnknown', 'Unflatten', 'UnflattenLike', 'Unsqueeze', 'Variable']


In [5]:
import inspect

from qewton.algorithms.building_blocks.array_operations import (
    DataConfiguration,
    Node,
    Unsqueeze,
    Squeeze,
)

print("=== DataConfiguration ===")
print(inspect.getsource(DataConfiguration))

print("=== Unsqueeze ===")
print(inspect.getsource(Unsqueeze))

print("=== Squeeze ===")
print(inspect.getsource(Squeeze))

print("=== Node ===")
print(inspect.getsource(Node))

=== DataConfiguration ===
class DataConfiguration(Serializable):
    """A *DataConfiguration* describes the expected structure of the data,
    including the axes and their dimensions, as well as the data type.
    It is used to ensure that the data being passed through the graph
    matches the expected format for different methods/algorithms.

    Args:
        *axes (Axes | EllipsisAxes): The axes that describe the
            structure of the data.
        dtype (_type_, optional): The datatype used in this
            configuration. Defaults to None.
        element_dtype (optional): The dtype of the elements of a tensor, a
            marker of :mod:`qewton.config.dtypes` such as `Float32`; None
            leaves it open.
    """

    #: The dtype of the elements of a tensor; None leaves it open.
    element_dtype: Any = None

    def __init__(self, *axes: Axes | EllipsisAxes, dtype=None, element_dtype=None):
        self.axes = axes
        self.dtype = dtype
        self.eleme

In [6]:
branch_input_port = data_loader.get_output_port(domain_var)

print("DataLoader output port:")
print(branch_input_port)

print("\nCNN branch port:")
print(model.branch_port)

print("\nDataLoader port attributes:")
print(vars(branch_input_port))

print("\nCNN branch port attributes:")
print(vars(model.branch_port))
import inspect

print(inspect.getsource(qewton.algorithms.CNNDeepONet))

sample = data["domain_mask"] if "data" in globals() else None


DataLoader output port:
DataLoader.domain

CNN branch port:
CNNDeepONet.x

DataLoader port attributes:
{'_static_data_configuration': BatchAxes([27]), GeometryAxes([100, 50, 25]), FeatureAxes([1]), 'node': <qewton.data.dataloaders.base.DataLoader object at 0x7f4a78226e30>, 'name': 'domain'}

CNN branch port attributes:
{'_static_data_configuration': BatchAxes([None]), FeatureAxes([1]), GeometryAxes([None, None, None]), 'node': <qewton.algorithms.dl_models.deeponet.deeponet_cnn.CNNDeepONet object at 0x7f4a782278b0>, 'name': 'x', 'default': <class 'qewton.graphs.nodes.NO_DEFAULT'>}
class CNNDeepONet(DeepONet[TensorType]):
    """A DeepONet architecture that uses fully connected networks (FCNs)
    for the trunk and a convolutional neural network (CNN) for the branch.
    The merge node can be configured to handle different output strategies,
    allowing for flexibility in how the outputs of the branch and trunk networks
    are combined. This DeepONet expects inputs of shape:
    - Bran

In [7]:
computation_graph = qewton.Graph()
transpose_node = qewton.bb.Transpose([0, 4, 1, 2, 3])
# Branch input
computation_graph.connect(
    data_loader.get_output_port(domain_var),
    transpose_node,
)

computation_graph.connect(
    transpose_node,
    model.branch_port,
)

# Trunk input
computation_graph.connect(
    data_loader.get_output_port(x_var),
    model.trunk_port,
)

# Model prediction
computation_graph.connect(
    model,
    temperature_constraint.input_1,
)

# Reference temperature
computation_graph.connect(
    data_loader.get_output_port(temperature_var),
    temperature_constraint.input_2,
)

In [8]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=250,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=[adam_phase],
    graphs=[computation_graph],
    training_objectives=[temperature_constraint],
    device="cuda:0",
)

trainer.run()

Optimization Phase 1: 100%|██████████| 250/250 [00:22<00:00, 11.18it/s, loss=0.71] 


In [ ]:
from load_data import x_var

# as an example, we use x3 (height component), you could also switch to x1 or x2

n_x3 = data_loader.get_output_port(domain_var).get_data_configuration(computation_graph).shape[3]
slice_x3 = qewton.viz.SliderAxis(x_var.leaves[2], init_state=n_x3 // 2)

layout = computation_graph.visualize(
    model.output_ports[0],
    reference=data_loader.get_output_port(temperature_var),
    mask=data_loader.get_output_port(domain_var),
    controls=[slice_x3],
    mode=qewton.optim.EvaluationPhase.TEST,
    z = x_var[2] # optional, embed in 3d and move slice with x3
)

fig = qewton.viz.Figure(layout)
app = qewton.viz.DashApplication.create(fig)
app.run(jupyter_mode='external')

Dash app running on http://127.0.0.1:8050/
